# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ruchitgoud/flyrankai-intern/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — Content age and performance

The research paper reports differences in content age between growing and declining content. Growing pages had an average age of about 185 days, while declining pages had an average age of about 228 days.

My methodology question is: does this analysis establish that older content causes performance to decline, or does it only show an observed association between content age and traffic direction?

I would want to check whether other factors could explain the difference and whether the validation design supports a causal interpretation.

### Finding 2 — Content freshness

The research paper reports a relationship between content freshness and growth-to-decline patterns, including differences between recently updated and stale content.

My methodology question is: does the analysis show that refreshing content causes better performance, or could other differences between refreshed and stale pages explain the observed result?

I would want to verify how the groups were constructed and whether the validation design separates the information available before an outcome from the outcome itself.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Why an honest split matters

My Week-5 model should be evaluated on data that does not give it an overly easy test. Because multiple content items can belong to the same client, a random row split can place rows from the same client in both training and test data.

I will compare the model using a random row split with a client-grouped split. In the grouped version, all rows from a client stay together, so the test set contains clients that were not used for training.

The grouped split is the more honest estimate for this use case because the model is intended to support decisions across clients it has not previously seen.

In [11]:
import os
import pandas as pd

repo_path = "/content/flyrankai-intern"

if not os.path.exists(repo_path):
    !git clone https://github.com/ruchitgoud/flyrankai-intern.git

%cd /content/flyrankai-intern

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Shape:", df.shape)
print("Columns:", len(df.columns))


/content/flyrankai-intern
Shape: (30000, 44)
Columns: 44


In [13]:
# Create the same proxy target used in Week 5
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print(df["is_declining_label"].value_counts())

is_declining_label
1    16262
0    13738
Name: count, dtype: int64


In [14]:
# Columns that must not be used as model features
exclude_cols = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

feature_cols = [
    col for col in df.columns
    if col not in exclude_cols
]

X = df[feature_cols]
y = df["is_declining_label"]
groups = df["client_id"]

print("Number of features:", len(feature_cols))
print("Number of clients:", groups.nunique())

Number of features: 40
Number of clients: 32


In [15]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# Random row split
X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

numeric_cols = X.select_dtypes(include="number").columns.tolist()
categorical_cols = X.select_dtypes(exclude="number").columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            numeric_cols
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_cols
        )
    ]
)

random_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(
        max_depth=5,
        random_state=42
    ))
])

random_model.fit(X_train_random, y_train_random)

random_prob = random_model.predict_proba(X_test_random)[:, 1]

random_roc_auc = roc_auc_score(y_test_random, random_prob)
random_ap = average_precision_score(y_test_random, random_prob)

print("BEFORE — Random row split")
print("ROC AUC:", round(random_roc_auc, 4))
print("Average Precision:", round(random_ap, 4))

BEFORE — Random row split
ROC AUC: 0.7933
Average Precision: 0.7682


In [16]:
from sklearn.model_selection import GroupShuffleSplit

# Client-grouped split
group_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    group_split.split(X, y, groups=groups)
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

group_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(
        max_depth=5,
        random_state=42
    ))
])

group_model.fit(X_train_group, y_train_group)

group_prob = group_model.predict_proba(X_test_group)[:, 1]

group_roc_auc = roc_auc_score(y_test_group, group_prob)
group_ap = average_precision_score(y_test_group, group_prob)

print("AFTER — Client-grouped split")
print("ROC AUC:", round(group_roc_auc, 4))
print("Average Precision:", round(group_ap, 4))

print("\nTraining clients:", groups_train.nunique())
print("Test clients:", groups_test.nunique())

print(
    "\nClient overlap:",
    len(set(groups_train) & set(groups_test))
)

AFTER — Client-grouped split
ROC AUC: 0.7233
Average Precision: 0.6746

Training clients: 25
Test clients: 7

Client overlap: 0


In [17]:
comparison = pd.DataFrame({
    "method": [
        "Week-5 reported result",
        "Week-6 random row split",
        "Week-6 client-grouped split"
    ],
    "roc_auc": [
        0.6392,
        random_roc_auc,
        group_roc_auc
    ],
    "average_precision": [
        0.6105,
        random_ap,
        group_ap
    ]
})

comparison

,method,roc_auc,average_precision
0,Week-5 reported result,0.639200,0.610500
1,Week-6 random row split,0.793294,0.768181
2,Week-6 client-grouped split,0.723279,0.674590


### Before vs after interpretation

The random row split allows rows from the same client to appear in both training and test data, so it can provide an easier evaluation.

The client-grouped split keeps each client entirely within either training or test data. This gives a more realistic test of whether the model can generalize to clients it did not see during training.

I therefore treat the client-grouped result as the more honest validation estimate for this decision-support task. Any change in performance between the two splits is useful evidence about how sensitive the model is to the validation design.

The results do not establish causal effects or prove that the model will improve page performance. They only measure predictive performance under the stated validation setups.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage audit

I will audit the features used by the Week-5 model to check whether any feature directly contains the target or information that would only be available after the prediction decision.

The target is derived from `trend_direction`, so `trend_direction` and `trend_pct` must not be used as model features. I will also check fields that may represent downstream decisions, future outcomes, or information created after the decision point.

The purpose of this audit is to make sure the model's validation score reflects genuine predictive signal rather than information leakage.

In [18]:
# Section 3 — Leakage audit

target_col = "is_declining_label"

# Potentially problematic columns
leakage_candidates = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "health_score",
    "priority_score",
    "action_type",
    "recommended_action"
]

print("Leakage candidates present in dataset:")
for col in leakage_candidates:
    if col in df.columns:
        print("-", col)

print("\nFeatures used by the model:")
print(feature_cols)

# Explicit checks
print("\nLeakage checks:")

print(
    "trend_direction used as feature:",
    "trend_direction" in feature_cols
)

print(
    "trend_pct used as feature:",
    "trend_pct" in feature_cols
)

print(
    "target used as feature:",
    target_col in feature_cols
)

print(
    "client_id used as feature:",
    "client_id" in feature_cols
)

Leakage candidates present in dataset:
- trend_direction
- trend_pct
- is_declining_label

Features used by the model:
['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier']

Leakage checks:
trend_direction used as feature: False
trend_pct used as feature: False
target used as feature: False
client_id used as feature: False


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Revised claim

The Week-5 Decision Tree achieved a higher ROC AUC and average precision than the Week-4 baseline under the evaluated validation setup. This indicates that the model provided a stronger predictive signal for the `down` proxy label on the held-out data.

However, this result does not show that the model will cause better traffic performance or that refreshing a page will improve its performance. The target is a proxy based on observed `trend_direction`, and the evaluation is limited to the available dataset and validation design.

The model should therefore be described as a decision-support and prioritization signal for identifying pages that may warrant human review, rather than as a causal recommendation or guarantee of improvement.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.